## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

### Установка поиска DuckDuckGo

Для замены OpenAI `WebSearchTool` используем пакет `ddgs`. Он выполняет обычный веб-поиск и не требует ключа OpenAI. Если пакет еще не установлен в окружении ноутбука, выполните следующую ячейку.

In [ ]:
%pip install -q ddgs

In [1]:
from agents import Agent, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from ddgs import DDGS
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
import importlib
import messenger
importlib.reload(messenger)
send_telegram = messenger.send_telegram

In [2]:
load_dotenv(override=True)

True

In [6]:
# Constants 

MODEL_NAME = "openai/gpt-4o-mini"

HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Telegram Agent: formats and sends the report in Telegram

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Стоимость hosted tools

Использование hosted tools может тарифицироваться отдельно от токенов модели. Актуальные условия и цены проверяйте в [документации OpenAI](https://developers.openai.com/api/docs/pricing). В примере ниже поиск выполняет DuckDuckGo через пакет `ddgs`, поэтому для самого поиска ключ OpenAI не нужен.


#### Локальная замена для этого ноутбука

Ниже `MyWebSearchTool` подключает поиск DuckDuckGo через `ddgs`: агент получает до пяти результатов с заголовками, URL и сниппетами. Это простой пример без ключа OpenAI; доступность и состав результатов зависят от поискового бэкенда. Для важных выводов открывайте источники и проверяйте их содержание.

In [7]:
@function_tool
async def MyWebSearchTool(query: str) -> str:
    """Ищет в интернете и возвращает заголовки, ссылки и краткие описания страниц.

    Args:
        query: Поисковый запрос на любом языке.
    """
    results = await asyncio.to_thread(DDGS().text, query, max_results=5)

    if not results:
        return f"По запросу {query!r} ничего не найдено."

    lines = []
    for number, item in enumerate(results, start=1):
        title = item.get("title") or "Без заголовка"
        url = item.get("href") or "Ссылка отсутствует"
        snippet = item.get("body") or "Описание отсутствует"
        lines.append(f"{number}. {title}\n   URL: {url}\n   Описание: {snippet}")

    return "\n\n".join(lines)

In [8]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results in 2-3 paragraphs and fewer than 300 words.
Use only the returned results, include source URLs next to relevant claims, and say when evidence is insufficient. Reply with the summary and sources.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [MyWebSearchTool]

In [12]:
import os

from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI


BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")

openrouter_client = AsyncOpenAI(base_url=BASE_URL, api_key=OPENROUTER_API_KEY)
model = OpenAIChatCompletionsModel(openai_client=openrouter_client, model=MODEL_NAME)

In [13]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=model, model_settings=settings)

In [14]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


In 2026, several AI agent frameworks have gained prominence, with **LangGraph**, **CrewAI**, **OpenAI Agents SDK**, and **Microsoft Agent Framework** leading the pack. LangGraph is recognized for handling complex workflows, supporting stateful multi-agent operations with features like retries and branching, making it ideal for businesses that require fine control over agent interactions ([LangChain](https://www.langchain.com/resources/ai-agent-frameworks), [LintedCode](https://lintedcode.com/ai-agent-orchestration-frameworks-2026-updates-langgraph-vs-microsoft-agent-framework-and-other-emerging-options/)). CrewAI, noted for its speed to market, allows quick deployment although it may incur higher costs in serverless architectures due to its heavier initial setup ([Gheware](https://devops.gheware.com/blog/posts/langgraph-vs-crewai-vs-autogen-comparison-2026.html)).

The **OpenAI Agents SDK** stands out as a user-friendly framework for building AI applications, integrating various AI models seamlessly for complex tasks while allowing maintenance of context across interactions ([OpenAI SDK Guide](https://developers.openai.com/api/docs/guides/agents/sdk)). Meanwhile, the **Microsoft Agent Framework**, which combines features from its predecessors, is designed for enterprise-scale deployment, offering robust state management and multi-agent workflow capabilities, aiding in scenarios that require human oversight ([Microsoft Learn](https://learn.microsoft.com/en-us/agent-framework/overview/), [NeuraPulse](https://neuraplus-ai.github.io/blog/ai-agent-frameworks-comparison-2026.html)).

Despite the popularity of these frameworks, precise user adoption metrics or specific case studies for 2026 remain scarce, suggesting further investigation or user feedback may be needed for a comprehensive understanding. For a thorough analysis, exploring features and performance trade-offs remains vital as enterprises assess how these frameworks align with their needs.

OPENAI_API_KEY is not set, skipping trace export


### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [15]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [16]:
WebSearchPlan.model_json_schema()


{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [17]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=model, output_type=WebSearchPlan)

In [18]:

result = await Runner.run(planner_agent, task)
result.final_output

OPENAI_API_KEY is not set, skipping trace export


WebSearchPlan(searches=[WebSearchItem(reason='To identify the leading AI Agent frameworks that gained popularity by 2026, analyzing trends and features in AI technology advancements.', query='most popular AI Agent frameworks 2026'), WebSearchItem(reason='To provide insights into the functionalities and use cases of AI Agent frameworks that emerged or evolved by 2026.', query='best AI Agent frameworks features 2026'), WebSearchItem(reason='To explore market reports and analyses that highlight trends and forecasted popular AI technologies and frameworks in 2026.', query='AI technology market trends 2026'), WebSearchItem(reason='To gather expert opinions and reviews on the effectiveness of AI Agent frameworks that are trending in 2026.', query='AI Agent frameworks reviews 2026'), WebSearchItem(reason='To find community discussions and recommendations related to AI Agent frameworks that are expected to dominate in 2026.', query='AI Agent frameworks community recommendations 2026')])

OPENAI_API_KEY is not set, skipping trace export


## Agent 3: The Writer Agent

In [22]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=model, output_type=ReportData)

## Agent 4: The Telegram agent

In [21]:
@function_tool
def send_telegram_tool(message: str) -> str:
    """Send the research report as a Telegram message.

    Args:
        message: Plain-text report message to send. The message must be under 4000 characters.
    """
    send_telegram(message)
    return "Telegram message sent successfully"


In [23]:
send_telegram_tool.params_json_schema

{'properties': {'message': {'description': 'Plain-text report message to send. Long messages are split into Telegram-safe parts.',
   'title': 'Message',
   'type': 'string'}},
 'required': ['message'],
 'title': 'send_telegram_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [26]:
INSTRUCTIONS = """
You are given a detailed research report. Make a concise plain-text Telegram message under 3500 characters: a short summary and the most important findings. Preserve key source URLs. Avoid HTML. Then use your tool to send the message.
"""

telegram_model_settings = ModelSettings(tool_choice="auto")


In [28]:
telegram_agent = Agent("Telegram sender", instructions=INSTRUCTIONS, model=model, tools=[send_telegram_tool], model_settings=telegram_model_settings)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [29]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and send it to Telegram

In [30]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_telegram(report: ReportData):
    print("Preparing Telegram message...")
    result = await Runner.run(telegram_agent, report.markdown_report, max_turns=3)
    print("Telegram message sent")
    return result.final_output

### Showtime!

In [31]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_telegram(report)  
    print("Hooray!")

Starting research...
Planning searches...


OPENAI_API_KEY is not set, skipping trace export


Will perform 5 searches


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Finished searching
Thinking about report...


OPENAI_API_KEY is not set, skipping trace export


Finished writing report
Preparing Telegram message...


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Telegram message sent
Hooray!


OPENAI_API_KEY is not set, skipping trace export


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>